# Compute residual norm coefficients

This notebook computes the residual norm coefficients as part of the variable weights.

In [1]:
import os
import yaml
import copy
import numpy as np
import xarray as xr

In [2]:
from scipy.stats import gmean

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

## ERA5 (x)

## WRF

In [11]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [12]:
N_levels = 12

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/C404_land/'
ds_example = xr.open_zarr(base_dir+'C404_SW_1980.zarr')
level = np.array(ds_example['bottom_top'])

In [13]:
# # get variable names
# varnames = list(conf['residual'].keys())
# varnames = varnames[:-5] # remove save_loc and others

# varname_upper = ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot', 'WRF_Q_tot_05']
# varname_surf = list(set(varnames) - set(varname_upper))

In [16]:
varname_upper = [
    'WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_W'
]

varname_surf = [
    'WRF_SP', 'WRF_MSLP', 'WRF_T2', 'WRF_TD2', 'WRF_U10', 
    'WRF_V10', 'WRF_PWAT_05'
]#

# 'WRF_precip_025', 'WRF_radar_composite_025', 'WRF_OLR', 'WRF_TCC', 'WRF_GLW', 'WRF_SWDOWN'

In [17]:
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['residual']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['residual']['prefix'], varname)
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['residual']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['residual']['prefix'], i_level, varname)
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [18]:
# separate upper air (list) and surf (float) std values
N_upper = len(varname_upper)
std_val_all = list(STD_values.values())
std_val_surf = np.array(std_val_all[:-N_upper])
std_val_upper = std_val_all[-N_upper:]

# combine
std_concat = np.concatenate([std_val_surf]+ std_val_upper)

# geometrical mean (not used)
std_g = gmean(np.sqrt(std_concat))

In [19]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std_6h = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data) / std_g
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [20]:
ds_std_6h.to_netcdf(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/C404_residual_1980_2019_12lev_clean.nc'
)

In [21]:
ds_GP = xr.open_dataset(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_residual_1980_2019_12lev_clean.nc'
)

ds_SW = xr.open_dataset(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_SW/mean_std/C404_residual_1980_2019_12lev_clean.nc'
)

for varname in ds_GP.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_GP[varname].values)
        print(ds_SW[varname].values)
    except:
        pass

=================== WRF_SP ===================
0.09474170269892271
0.14336751449395502
=================== WRF_MSLP ===================
0.10553652852635395
0.1587170411314873
=================== WRF_T2 ===================
0.7590723050331437
0.6732129231471389
=================== WRF_TD2 ===================
0.6361887595442969
0.5409302405061918
=================== WRF_U10 ===================
2.5823118138177117
1.8848941542568398
=================== WRF_V10 ===================
2.1854107436389905
1.8462435186973585
=================== WRF_PWAT_05 ===================
0.6827033851674961
0.7153365381851537
=================== WRF_P ===================
[0.09463757 0.09471565 0.09495939 0.09549486 0.09649085 0.09769533
 0.0985836  0.09898395 0.09906351 0.09953854 0.10053633 0.10540999]
[0.14333307 0.14366548 0.14434705 0.1456116  0.14784524 0.1508469
 0.15362882 0.15536602 0.15601627 0.15738414 0.15926835 0.16673692]
=================== WRF_U ===================
[2.61295577 2.17632909 1.910400